# Adding neuromodulation to an SNN, one small step at a time

This notebook is the readable companion to the compact `neuromod_snn/` package.

The point is simple: if you already have an SNN, adding a basic ANN neuromodulator only needs three additions:

1. Define an MLP.
2. Add `modulator.parameters()` to the optimizer.
3. In the SNN running loop, feed current SNN state into the modulator and use its output as the new SNN parameters.

We start with real SHD data, train a small SNN, then add the simplest possible parameter-substitution modulator. Only after that do we add optional ideas: additive modulation, choosing modulator inputs, reducing modulator size, bottleneck neuromodulators, and SNN modulation.

The defaults intentionally use a small subset of SHD and a small number of epochs so the notebook is fast enough to follow.

## Base SNN equations

The hidden layer is a recurrent leaky integrate-and-fire layer. For input spikes `x_t`, hidden spikes `z_t`, synaptic current `s_t`, membrane `u_t`, threshold `theta`, reset `u_reset`, and rest potential `u_rest`:

$$
I_t = x_t W_{in} + z_{t-1} W_{rec}
$$

$$
s_t = \alpha_1 s_{t-1} + I_t
$$

$$
z_t = H(u_t - \theta)
$$

$$
u_{t+1} = \beta_1 (u_t - u_{rest}) + u_{rest} + (1 - \beta_1)s_t - z_t(\theta - u_{reset})
$$

The readout is non-spiking:

$$
f_t = \alpha_2 f_{t-1} + z_t W_{out}
$$

$$
y_t = \beta_2 y_{t-1} + (1 - \beta_2)f_t
$$

Training uses cross-entropy on `max_t y_t` and a surrogate gradient for the spike function.

In [ ]:
import os
import math
import random
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import h5py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype = torch.float32
print('device:', device)

# Real SHD geometry.
nb_inputs = 700
nb_outputs = 20

# Reduced tutorial settings. Increase these for real experiments.
nb_hidden = 128
nb_steps = 100
max_time = 1.4
batch_size = 32
train_samples = 512
test_samples = 256
epochs_snn = 2
epochs_mod = 2
lr = 2e-3
mod_interval = 10

time_step = 1e-3
tau_syn = 10e-3
tau_mem = 20e-3

## Load real SHD

This expects the standard SHD HDF5 files used by the original experiments:

```text
~/data/hdspikes/shd_train.h5
~/data/hdspikes/shd_test.h5
```

Each file contains `spikes/times`, `spikes/units`, and `labels`. The generator below bins events into dense tensors with shape `[batch, time, input_channel]`.

In [ ]:
cache_dir = Path(os.path.expanduser('~/data'))
cache_subdir = 'hdspikes'
train_path = cache_dir / cache_subdir / 'shd_train.h5'
test_path = cache_dir / cache_subdir / 'shd_test.h5'

if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError(
        f'SHD files not found. Expected:\n  {train_path}\n  {test_path}\n'
        'If needed, download/create SHD first, then rerun this cell.'
    )

train_h5 = h5py.File(train_path, 'r')
test_h5 = h5py.File(test_path, 'r')
x_train, y_train = train_h5['spikes'], train_h5['labels']
x_test, y_test = test_h5['spikes'], test_h5['labels']

print('train samples:', len(y_train), 'test samples:', len(y_test))
print('spike fields:', list(x_train.keys()))

In [ ]:
def shd_dense_batches(X, y, batch_size, nb_steps, nb_units, max_time, max_samples=None, shuffle=True):
    labels = np.asarray(y, dtype=np.int64)
    sample_index = np.arange(len(labels))
    if max_samples is not None:
        sample_index = sample_index[:min(int(max_samples), len(sample_index))]
    if shuffle:
        np.random.shuffle(sample_index)

    firing_times = X['times']
    units_fired = X['units']
    time_bins = np.linspace(0.0, max_time, num=nb_steps + 1)

    for start in range(0, len(sample_index), batch_size):
        batch_index = sample_index[start:start + batch_size]
        if len(batch_index) == 0:
            continue
        dense = torch.zeros((len(batch_index), nb_steps, nb_units), dtype=dtype)
        target = torch.tensor(labels[batch_index], dtype=torch.long)

        for b, idx in enumerate(batch_index):
            times = np.digitize(firing_times[idx], time_bins) - 1
            times = np.clip(times, 0, nb_steps - 1).astype(np.int64)
            units = np.asarray(units_fired[idx], dtype=np.int64)
            keep = (units >= 0) & (units < nb_units)
            dense[b, times[keep], units[keep]] = 1.0

        yield dense.to(device), target.to(device)

# Quick visual check.
xb, yb = next(shd_dense_batches(x_train, y_train, batch_size=4, nb_steps=nb_steps, nb_units=nb_inputs, max_time=max_time, max_samples=4, shuffle=False))
print('batch:', tuple(xb.shape), 'labels:', yb.tolist())

if plt is not None:
    ts, cs = torch.where(xb[0].cpu() > 0)
    plt.figure(figsize=(7, 2.5))
    plt.scatter(ts, cs, s=3)
    plt.title(f'SHD sample, label={int(yb[0])}')
    plt.xlabel('time bin')
    plt.ylabel('input channel')
    plt.tight_layout()

## Baseline SNN code

This is the normal SNN. No modulation yet.

In [ ]:
class SpikeFn(torch.autograd.Function):
    scale = 25.0

    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return (x > 0).to(x.dtype)

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        return grad_output / (SpikeFn.scale * x.abs() + 1.0).pow(2)

spike_fn = SpikeFn.apply

PARAM_RANGES = {
    'alpha_1': (1.0 / math.e, 0.995),
    'beta_1': (1.0 / math.e, 0.995),
    'alpha_2': (1.0 / math.e, 0.995),
    'beta_2': (1.0 / math.e, 0.995),
    'thr': (0.5, 1.5),
    'reset': (-0.5, 0.5),
    'rest': (-0.5, 0.5),
}

HIDDEN_PARAMS = ['alpha_1', 'beta_1', 'thr', 'reset', 'rest']
OUTPUT_PARAMS = ['alpha_2', 'beta_2']
ALL_PARAM_NAMES = HIDDEN_PARAMS + OUTPUT_PARAMS


def init_snn():
    alpha = math.exp(-time_step / tau_syn)
    beta = math.exp(-time_step / tau_mem)
    state = {
        'w1': nn.Parameter(torch.randn(nb_inputs, nb_hidden, device=device) * 0.4 / math.sqrt(nb_inputs)),
        'v1': nn.Parameter(torch.randn(nb_hidden, nb_hidden, device=device) * 0.4 / math.sqrt(nb_hidden)),
        'w2': nn.Parameter(torch.randn(nb_hidden, nb_outputs, device=device) * 0.4 / math.sqrt(nb_hidden)),
        'alpha_1': nn.Parameter(torch.full((1, nb_hidden), alpha, device=device)),
        'beta_1': nn.Parameter(torch.full((1, nb_hidden), beta, device=device)),
        'thr': nn.Parameter(torch.ones(1, nb_hidden, device=device)),
        'reset': nn.Parameter(torch.zeros(1, nb_hidden, device=device)),
        'rest': nn.Parameter(torch.zeros(1, nb_hidden, device=device)),
        'alpha_2': nn.Parameter(torch.full((1, nb_outputs), alpha, device=device)),
        'beta_2': nn.Parameter(torch.full((1, nb_outputs), beta, device=device)),
    }
    return state


def state_parameters(state: Dict[str, nn.Parameter]) -> List[nn.Parameter]:
    return [p for p in state.values() if p.requires_grad]


def clamp_snn_state_(state):
    with torch.no_grad():
        for name in ALL_PARAM_NAMES:
            lo, hi = PARAM_RANGES[name]
            state[name].clamp_(float(lo), float(hi))

In [ ]:
def run_snn(inputs, state, return_records=False):
    B = inputs.size(0)
    x = inputs.to(device)

    w1, v1, w2 = state['w1'], state['v1'], state['w2']
    alpha_1 = state['alpha_1'].expand(B, nb_hidden)
    beta_1 = state['beta_1'].expand(B, nb_hidden)
    thr = state['thr'].expand(B, nb_hidden)
    reset = state['reset'].expand(B, nb_hidden)
    rest = state['rest'].expand(B, nb_hidden)
    alpha_2 = state['alpha_2'].expand(B, nb_outputs)
    beta_2 = state['beta_2'].expand(B, nb_outputs)

    syn = torch.zeros(B, nb_hidden, device=device)
    mem = torch.zeros(B, nb_hidden, device=device)
    spk = torch.zeros(B, nb_hidden, device=device)
    flt = torch.zeros(B, nb_outputs, device=device)
    out = torch.zeros(B, nb_outputs, device=device)

    spk_rec, out_rec = [], []
    h1_from_input = torch.einsum('btc,ch->bth', x, w1)

    for t in range(nb_steps):
        h1 = h1_from_input[:, t] + torch.einsum('bh,hk->bk', spk, v1)
        mthr = mem - thr
        spk = spike_fn(mthr)
        rst = (mthr > 0).to(dtype)

        syn = alpha_1 * syn + h1
        mem = beta_1 * (mem - rest) + rest + (1.0 - beta_1) * syn - rst * (thr - reset)

        h2 = torch.einsum('bh,ho->bo', spk, w2)
        flt = alpha_2 * flt + h2
        out = beta_2 * out + (1.0 - beta_2) * flt

        spk_rec.append(spk)
        out_rec.append(out)

    out_rec = torch.stack(out_rec, dim=1)
    logits = out_rec.max(dim=1).values
    if return_records:
        return logits, torch.stack(spk_rec, dim=1), out_rec
    return logits

In [ ]:
def accuracy(logits, target):
    return (logits.argmax(dim=1) == target).float().mean().item()


def evaluate_snn(state, max_samples=test_samples):
    total_loss, total_acc, n = 0.0, 0.0, 0
    with torch.no_grad():
        for xb, yb in shd_dense_batches(x_test, y_test, batch_size, nb_steps, nb_inputs, max_time, max_samples=max_samples, shuffle=False):
            logits = run_snn(xb, state)
            loss = F.cross_entropy(logits, yb)
            total_loss += loss.item() * len(yb)
            total_acc += accuracy(logits, yb) * len(yb)
            n += len(yb)
    return total_loss / n, total_acc / n


def train_snn(state):
    optimizer = torch.optim.Adam(state_parameters(state), lr=lr)
    for epoch in range(1, epochs_snn + 1):
        for xb, yb in shd_dense_batches(x_train, y_train, batch_size, nb_steps, nb_inputs, max_time, max_samples=train_samples, shuffle=True):
            logits = run_snn(xb, state)
            loss = F.cross_entropy(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            clamp_snn_state_(state)
        test_loss, test_acc = evaluate_snn(state)
        print(f'SNN epoch {epoch:02d} | test loss {test_loss:.3f} | test acc {test_acc:.3f}')
    return state

snn_state = train_snn(init_snn())

## Add the simplest ANN modulator: substitution

This is the part that should feel easy.

A substitution modulator is just:

$$
p_t^{new} = f_\phi(p_t)
$$

where `p_t` is the vector of current SNN parameters. The MLP output is mapped into valid parameter ranges, then used inside the SNN loop.

So the implementation change is only:

1. **MLP:** define `ParameterMLP`.
2. **Optimizer:** add `modulator.parameters()`.
3. **Running loop:** feed parameters into the MLP, then replace the local parameters with the MLP output.

The next two cells intentionally avoid masks, groups, bottlenecks, and compression. Those are useful later, but they are not the core idea.

In [ ]:
def pack_params(params: Dict[str, torch.Tensor]) -> torch.Tensor:
    return torch.cat([params[name] for name in ALL_PARAM_NAMES], dim=1)


def split_params(flat: torch.Tensor) -> Dict[str, torch.Tensor]:
    sizes = {
        'alpha_1': nb_hidden, 'beta_1': nb_hidden, 'thr': nb_hidden, 'reset': nb_hidden, 'rest': nb_hidden,
        'alpha_2': nb_outputs, 'beta_2': nb_outputs,
    }
    out, start = {}, 0
    for name in ALL_PARAM_NAMES:
        end = start + sizes[name]
        raw = flat[:, start:end]
        lo, hi = PARAM_RANGES[name]
        # Sigmoid gives raw in [0, 1]. This line maps each output into its legal range:
        # thr: 0.5 + raw * 1.0, reset/rest: -0.5 + raw * 1.0, alpha/beta: decay range.
        out[name] = float(lo) + raw * (float(hi) - float(lo))
        start = end
    return out


class ParameterMLP(nn.Module):
    def __init__(self, hidden_size=128):
        super().__init__()
        param_dim = 5 * nb_hidden + 2 * nb_outputs
        self.net = nn.Sequential(
            nn.Linear(param_dim, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, param_dim),
            nn.Sigmoid(),  # substitution: output is in [0, 1], then mapped into legal parameter ranges
        )

    def forward(self, params: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        return split_params(self.net(pack_params(params)))

In [ ]:
def run_snn_with_substitution_modulator(inputs, state, modulator):
    B = inputs.size(0)
    x = inputs.to(device)

    w1, v1, w2 = state['w1'], state['v1'], state['w2']

    # NEW FOR MODULATOR: local per-sample parameters that the MLP is allowed to replace.
    params = {name: state[name].expand(B, -1).clone() for name in ALL_PARAM_NAMES}

    syn = torch.zeros(B, nb_hidden, device=device)
    mem = torch.zeros(B, nb_hidden, device=device)
    spk = torch.zeros(B, nb_hidden, device=device)
    flt = torch.zeros(B, nb_outputs, device=device)
    out = torch.zeros(B, nb_outputs, device=device)

    spk_rec, out_rec = [], []
    h1_from_input = torch.einsum('btc,ch->bth', x, w1)

    for t in range(nb_steps):
        h1 = h1_from_input[:, t] + torch.einsum('bh,hk->bk', spk, v1)
        mthr = mem - params['thr']
        spk = spike_fn(mthr)
        rst = (mthr > 0).to(dtype)

        syn = params['alpha_1'] * syn + h1
        mem = params['beta_1'] * (mem - params['rest']) + params['rest'] + (1.0 - params['beta_1']) * syn - rst * (params['thr'] - params['reset'])

        h2 = torch.einsum('bh,ho->bo', spk, w2)
        flt = params['alpha_2'] * flt + h2
        out = params['beta_2'] * out + (1.0 - params['beta_2']) * flt

        spk_rec.append(spk)
        out_rec.append(out)

        if t % mod_interval == 0:
            # NEW FOR MODULATOR: input to modulator is the current SNN parameters.
            # NEW FOR MODULATOR: output from modulator is the new SNN parameters.
            params = modulator(params)

    out_rec = torch.stack(out_rec, dim=1)
    return out_rec.max(dim=1).values, (torch.stack(spk_rec, dim=1), out_rec)

In [ ]:
def clone_state(state, requires_grad=True):
    return {k: nn.Parameter(v.detach().clone(), requires_grad=requires_grad) for k, v in state.items()}


def evaluate_sub_mod(state, modulator, max_samples=test_samples):
    total_loss, total_acc, n = 0.0, 0.0, 0
    modulator.eval()
    with torch.no_grad():
        for xb, yb in shd_dense_batches(x_test, y_test, batch_size, nb_steps, nb_inputs, max_time, max_samples=max_samples, shuffle=False):
            logits, _ = run_snn_with_substitution_modulator(xb, state, modulator)
            loss = F.cross_entropy(logits, yb)
            total_loss += loss.item() * len(yb)
            total_acc += accuracy(logits, yb) * len(yb)
            n += len(yb)
    modulator.train()
    return total_loss / n, total_acc / n


def train_substitution_modulator(base_state):
    state = clone_state(base_state, requires_grad=True)
    modulator = ParameterMLP(hidden_size=128).to(device)

    # NEW FOR MODULATOR: same optimizer as before, plus the MLP parameters.
    optimizer = torch.optim.Adam(state_parameters(state) + list(modulator.parameters()), lr=lr)

    for epoch in range(1, epochs_mod + 1):
        for xb, yb in shd_dense_batches(x_train, y_train, batch_size, nb_steps, nb_inputs, max_time, max_samples=train_samples, shuffle=True):
            logits, (spks, _) = run_snn_with_substitution_modulator(xb, state, modulator)
            loss = F.cross_entropy(logits, yb) + 1e-6 * spks.sum()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            clamp_snn_state_(state)
        test_loss, test_acc = evaluate_sub_mod(state, modulator)
        print(f'substitution mod epoch {epoch:02d} | test loss {test_loss:.3f} | test acc {test_acc:.3f}')
    return state, modulator

sub_state, sub_modulator = train_substitution_modulator(snn_state)

## Addition instead of substitution

Substitution replaces the parameter:

$$
p_t \leftarrow f_\phi(p_t)
$$

Addition predicts a small change:

$$
p_t \leftarrow \operatorname{clip}(p_t + \Delta p_t, p_{min}, p_{max})
$$

This is usually a one-line conceptual change: the MLP uses `Tanh` instead of `Sigmoid`, and the running loop adds the output instead of replacing the parameter.

In [ ]:
def apply_delta(params, deltas, scale=0.05):
    updated = {}
    for name in ALL_PARAM_NAMES:
        lo, hi = PARAM_RANGES[name]
        width = float(hi) - float(lo)
        updated[name] = (params[name] + scale * width * deltas[name]).clamp(float(lo), float(hi))
    return updated


class AdditiveParameterMLP(nn.Module):
    def __init__(self, hidden_size=128):
        super().__init__()
        param_dim = 5 * nb_hidden + 2 * nb_outputs
        self.sizes = {'alpha_1': nb_hidden, 'beta_1': nb_hidden, 'thr': nb_hidden, 'reset': nb_hidden, 'rest': nb_hidden,
                      'alpha_2': nb_outputs, 'beta_2': nb_outputs}
        self.net = nn.Sequential(nn.Linear(param_dim, hidden_size), nn.SiLU(), nn.Linear(hidden_size, param_dim), nn.Tanh())

    def forward(self, params):
        flat = self.net(pack_params(params))
        out, start = {}, 0
        for name in ALL_PARAM_NAMES:
            end = start + self.sizes[name]
            out[name] = flat[:, start:end]
            start = end
        return out


def run_snn_with_additive_modulator(inputs, state, modulator):
    B = inputs.size(0)
    x = inputs.to(device)
    w1, v1, w2 = state['w1'], state['v1'], state['w2']
    params = {name: state[name].expand(B, -1).clone() for name in ALL_PARAM_NAMES}

    syn = torch.zeros(B, nb_hidden, device=device)
    mem = torch.zeros(B, nb_hidden, device=device)
    spk = torch.zeros(B, nb_hidden, device=device)
    flt = torch.zeros(B, nb_outputs, device=device)
    out = torch.zeros(B, nb_outputs, device=device)
    spk_rec, out_rec = [], []
    h1_from_input = torch.einsum('btc,ch->bth', x, w1)

    for t in range(nb_steps):
        h1 = h1_from_input[:, t] + torch.einsum('bh,hk->bk', spk, v1)
        mthr = mem - params['thr']
        spk = spike_fn(mthr)
        rst = (mthr > 0).to(dtype)
        syn = params['alpha_1'] * syn + h1
        mem = params['beta_1'] * (mem - params['rest']) + params['rest'] + (1.0 - params['beta_1']) * syn - rst * (params['thr'] - params['reset'])
        h2 = torch.einsum('bh,ho->bo', spk, w2)
        flt = params['alpha_2'] * flt + h2
        out = params['beta_2'] * out + (1.0 - params['beta_2']) * flt
        spk_rec.append(spk)
        out_rec.append(out)

        if t % mod_interval == 0:
            # NEW FOR MODULATOR: MLP emits deltas instead of replacement values.
            params = apply_delta(params, modulator(params))

    out_rec = torch.stack(out_rec, dim=1)
    return out_rec.max(dim=1).values, (torch.stack(spk_rec, dim=1), out_rec)

## Choosing what the modulator sees

Once the simple version is clear, choosing inputs is easy. Build the modulator input from a list of tensors. To ablate an input, comment it out and adjust the MLP input size.

For example, the first modulator used only parameters. Later you might add recent hidden spikes:

```python
mod_input = torch.cat([
    params['alpha_1'],
    params['beta_1'],
    params['thr'],
    # params['reset'],     # comment out to remove
    # params['rest'],      # comment out to remove
    params['alpha_2'],
    params['beta_2'],
    hid_spike_count,       # recent activity signal
], dim=1)
```

This is what the full script exposes as `--ann_in_disable`.

In [ ]:
# Small helper for experiments after the simple version.
# Comment names out of this list to remove them from the modulator input.
MODULATOR_INPUTS = [
    'alpha_1',
    'beta_1',
    'thr',
    # 'reset',
    # 'rest',
    'alpha_2',
    'beta_2',
]

input_dim = sum(nb_hidden if name in HIDDEN_PARAMS else nb_outputs for name in MODULATOR_INPUTS)
print('modulator input dim with selected inputs:', input_dim)

## Making the modulator smaller

The modulator can grow because it reads or writes per-neuron parameters. The simplest reductions are:

1. Smaller MLP hidden layer.
2. Fewer input blocks.
3. Fewer output blocks.
4. Shared modulation across groups of neurons.

In the full code these correspond to options like `ann_in_disable`, `ann_out_disable`, `group_size`, `mod_fixed_mask_enable`, and `channel_compress_*`.

In [ ]:
def mlp_param_count(input_dim, hidden_dim, output_dim):
    return (input_dim * hidden_dim + hidden_dim) + (hidden_dim * output_dim + output_dim)

full_dim = 5 * nb_hidden + 2 * nb_outputs
small_input_dim = input_dim
full_output_dim = full_dim
threshold_only_output_dim = nb_hidden
hidden_group_size = 8
grouped_threshold_output_dim = math.ceil(nb_hidden / hidden_group_size)

print('full parameter-input modulator:', mlp_param_count(full_dim, 128, full_output_dim))
print('smaller inputs, threshold only:', mlp_param_count(small_input_dim, 64, threshold_only_output_dim))
print('grouped threshold only:        ', mlp_param_count(small_input_dim, 64, grouped_threshold_output_dim))

## More biological versions

After the easy version, the biologically motivated versions are just constraints on the same pattern.

A neuromodulator bottleneck is:

$$
f_\phi(\text{SNN state}) \rightarrow \eta_t
$$

$$
g_\psi(\eta_t) \rightarrow \Delta p_t
$$

where `eta_t` has far fewer dimensions than the full parameter vector.

An SNN modulator replaces the ANN MLP with another spiking network:

$$
z^m_t = H(u^m_t - \theta_m), \qquad \Delta p_t = z^m_t W_{decode}
$$

The important message is that the interface stays the same: input is SNN state/activity, output is parameter values or parameter deltas.

In [ ]:
class TinySNNModulator(nn.Module):
    def __init__(self, input_dim, output_dim, hidden=64):
        super().__init__()
        self.w_in = nn.Linear(input_dim, hidden, bias=False)
        self.w_out = nn.Linear(hidden, output_dim)
        self.alpha = 0.9
        self.beta = 0.85
        self.threshold = 1.0

    def forward(self, feature_sequence):
        B, T, _ = feature_sequence.shape
        syn = torch.zeros(B, self.w_in.out_features, device=feature_sequence.device)
        mem = torch.zeros_like(syn)
        spike_sum = torch.zeros_like(syn)
        for t in range(T):
            syn = self.alpha * syn + self.w_in(feature_sequence[:, t])
            spk = spike_fn(mem - self.threshold)
            mem = self.beta * mem + (1.0 - self.beta) * syn - spk * self.threshold
            spike_sum = spike_sum + spk
        return torch.tanh(self.w_out(spike_sum / T))

# Shape check only. This is the same interface: features in, parameter effects out.
example_feature = torch.zeros(batch_size, input_dim, device=device)
feature_sequence = example_feature.unsqueeze(1).repeat(1, mod_interval, 1)
snn_mod = TinySNNModulator(input_dim=input_dim, output_dim=grouped_threshold_output_dim).to(device)
print('SNN modulator input:', tuple(feature_sequence.shape))
print('SNN modulator output:', tuple(snn_mod(feature_sequence).shape))

## Where this lives in the full runner

Search these names in the release package if you want the full training runner:

- baseline SNN: `setup_model`, `run_snn_hetero`, `train_snn_hetero`
- ANN substitution/addition: `ModulatingMLP`, `run_snn_modulated`, `mlp_sub`, `mlp_add`
- input/output selection: `ann_in_disable`, `ann_out_disable`
- size reduction: `group_size`, `mod_fixed_mask_enable`, `channel_compress_*`
- neuromodulator bottleneck: `NeuromodulatorMapper`, `nm_enable`, `nm_counts`
- SNN modulation: `SNNAdditiveModulator`, `SNNSubstitutionModulator`, `snn_add`, `snn_sub`

The full script is necessarily bigger because it handles sweeps, checkpoints, validation, fixed masks, grouped modulation, compression, SNN modulators, and neuromodulator mappers. But the core idea remains the three-line teaching version above.